In [ ]:
# Standard setup — adds project root to path
import sys, os, subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DBT_PROJECT = PROJECT_ROOT / "medical_warehouse"
print("Project root:", PROJECT_ROOT)
print("dbt project: ", DBT_PROJECT)

In [ ]:
# Reads all JSON files from data/raw/telegram_messages/ and inserts into
# the raw.telegram_messages table in PostgreSQL.
# Make sure docker compose up -d has been run first.
from src.load_to_postgres import get_engine, create_raw_table, load_json_files

engine = get_engine()
create_raw_table(engine)
total = load_json_files(engine, str(PROJECT_ROOT / "data"))
print(f"Loaded {total} messages into raw.telegram_messages")

In [ ]:
# Quickly checks the raw table loaded correctly before running dbt.
import pandas as pd
from src.load_to_postgres import get_engine

engine = get_engine()
df = pd.read_sql("SELECT channel_name, COUNT(*) as n FROM raw.telegram_messages GROUP BY channel_name", engine)
print("Rows per channel in raw.telegram_messages:")
print(df.to_string(index=False))

In [ ]:
# Runs all dbt models: staging first, then marts.
# Equivalent to: dbt run (from inside the medical_warehouse folder)
result = subprocess.run(
    ["dbt", "run"],
    cwd=str(DBT_PROJECT),
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERRORS:\n", result.stderr)

In [ ]:
# Inspect the cleaned staging layer output.
import pandas as pd
from src.load_to_postgres import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM staging.stg_telegram_messages LIMIT 10", engine)
print(df[["message_id","channel_name","message_date","message_length","has_image"]].to_string(index=False))

In [ ]:
# Inspect the final star schema mart tables.
from src.load_to_postgres import get_engine
import pandas as pd

engine = get_engine()

print("=== dim_channels ===")
print(pd.read_sql("SELECT * FROM marts.dim_channels", engine).to_string(index=False))

print("\n=== dim_dates (sample) ===")
print(pd.read_sql("SELECT * FROM marts.dim_dates LIMIT 5", engine).to_string(index=False))

print("\n=== fct_messages (sample) ===")
print(pd.read_sql("SELECT * FROM marts.fct_messages LIMIT 5", engine).to_string(index=False))